# Execução no Azure Machine Learning

Este notebook registra os recursos lógicos e submete o otimizador ao Azure ML. A infraestrutura deve existir previamente, pelo Terraform em `infra/azure` ou pelo portal. A criação e execução de recursos pode gerar cobranças.

## 1. Preparação do SDK

As bibliotecas são instaladas no próprio kernel, sem um terminal separado.

In [ ]:
from pathlib import Path
import subprocess, sys
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'azure-ai-ml>=1.24,<2', 'azure-identity>=1.17,<2', '--quiet'])
print('SDK preparado.')

## 2. Identificação do workspace

Preencha os valores exibidos pelo portal ou pelos outputs do Terraform.

In [ ]:
SUBSCRIPTION_ID = 'substitua-pelo-id-da-assinatura'
RESOURCE_GROUP = 'substitua-pelo-resource-group'
WORKSPACE_NAME = 'substitua-pelo-workspace'
assert not any(v.startswith('substitua') for v in (SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE_NAME)), 'Preencha os dados do workspace.'

## 3. Autenticação

Em uma Compute Instance, a credencial padrão costuma ser suficiente. Fora do Azure, o navegador será aberto para autenticação.

In [ ]:
from azure.ai.ml import MLClient, load_data, load_environment, load_job
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
try:
    credential = DefaultAzureCredential()
    credential.get_token('https://management.azure.com/.default')
except Exception:
    credential = InteractiveBrowserCredential()
client = MLClient(credential, SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE_NAME)
print('Workspace:', client.workspaces.get(WORKSPACE_NAME).name)

## 4. Ambiente e ativo de dados

O ambiente define Python, visualização e MLflow. O cenário JSON é versionado como ativo de dados.

In [ ]:
environment = load_environment(source=ROOT / 'azure' / 'environment.yml')
registered_environment = client.environments.create_or_update(environment)
data = load_data(source=ROOT / 'azure' / 'data.yml')
registered_data = client.data.create_or_update(data)
print('Ambiente:', registered_environment.name, registered_environment.version)
print('Dados:', registered_data.name, registered_data.version)

## 5. Job serverless

A ausência de `compute` no manifesto faz o Azure ML alocar recursos sob demanda e liberá-los ao final.

In [ ]:
job = load_job(source=ROOT / 'azure' / 'job.yml')
submitted_job = client.jobs.create_or_update(job)
print('Job:', submitted_job.name)
print('Studio:', submitted_job.studio_url)
client.jobs.stream(submitted_job.name)

## 6. Sweep de hiperparâmetros

O sweep compara população, crossover, mutação e elitismo, minimizando `fitness`. Ative somente quando desejar executar as doze tentativas.

In [ ]:
EXECUTAR_SWEEP = False
if EXECUTAR_SWEEP:
    sweep = load_job(source=ROOT / 'azure' / 'sweep.yml')
    submitted_sweep = client.jobs.create_or_update(sweep)
    print('Sweep:', submitted_sweep.name)
    print('Studio:', submitted_sweep.studio_url)
else:
    print('Sweep não submetido. Altere EXECUTAR_SWEEP para True quando necessário.')

## 7. Encerramento e custos

Confirme que o `cpu-cluster` retornou a zero nós e desligue qualquer Compute Instance. Quando os recursos não forem mais necessários, remova o Resource Group pelo portal ou destrua a infraestrutura Terraform.